# Financial RAG — End-to-End Demo

本 Notebook 展示金融 RAG 系统的完整流程，适合面试演示和快速理解。

```bash
# 运行前确保已安装依赖
pip install -r requirements.txt
# 并配置 .env 中的 MIMO_API_KEY
```

## 1. 系统配置加载

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from src.config import Config

config = Config()
print(f"LLM: {config.llm.model}")
print(f"Embedding: {config.embedding.model}")
print(f"Hybrid: {config.hybrid.strategy}")
print(f"Chunk size: {config.chunker.chunk_size}")
print(f"API Key: {'*' * 8 + config.api_key[-4:] if config.api_key else 'NOT SET'}")

## 2. 构建检索组件

In [ ]:
from src.embeddings.siliconflow_embedder import SiliconFlowEmbedder
from src.vectorstore.chroma_store import ChromaStore
from src.retriever.retriever import Retriever, RetrieverConfig
from src.retriever.hybrid_retriever import HybridRetriever
from src.retriever.bm25_retriever import BM25Retriever

# Embedding + VectorStore
embedder = SiliconFlowEmbedder(api_key=config.api_key, model=config.embedding.model)
store = ChromaStore(
    persist_directory=config.vectorstore.persist_directory,
    collection_name=config.vectorstore.collection_name,
)

# Stats
stats = store.get_stats()
print(f"Indexed documents: {stats['document_count']}")
print(f"Collection: {config.vectorstore.collection_name}")

## 3. 混合检索演示

In [ ]:
# Build retrievers
base_retriever = Retriever(
    embedder, store,
    RetrieverConfig(top_k=config.hybrid.vector_fetch_k, score_threshold=0.0),
)
bm25 = BM25Retriever(store)
hybrid = HybridRetriever(
    retriever=base_retriever,
    bm25_retriever=bm25,
    config=config.hybrid,
    score_threshold=config.retriever.score_threshold,
)

# Query
query = "什么是市盈率？"
print(f"Query: {query}\n")

# Vector search
from dataclasses import replace
vector_results = base_retriever.retrieve(query)
print(f"=== Vector Search ({len(vector_results)} results) ===")
for i, r in enumerate(vector_results[:3]):
    title = r.metadata.get('title', r.metadata.get('source', 'N/A'))
    print(f"  [{i+1}] score={r.score:.4f} | {title}")
    print(f"      {r.content[:100]}...")

# Hybrid search
hybrid_results = hybrid.retrieve(query)
print(f"\n=== Hybrid Search ({len(hybrid_results)} results) ===")
for i, r in enumerate(hybrid_results[:3]):
    title = r.metadata.get('title', r.metadata.get('source', 'N/A'))
    print(f"  [{i+1}] score={r.score:.4f} | {title}")
    print(f"      {r.content[:100]}...")

## 4. RAG Pipeline 查询

In [ ]:
from src.generator.mimo_llm import MimoLLM
from src.rag_pipeline import RAGPipeline

llm = MimoLLM(
    api_key=config.api_key,
    model=config.llm.model,
    base_url=config.llm.base_url,
    temperature=0.7,
    max_tokens=2048,
)

pipeline = RAGPipeline(
    retriever=hybrid,
    llm=llm,
    rag_config=config.rag,
)

# Query
question = "什么是GDP？请详细解释"
print(f"Q: {question}\n")

result = pipeline.query(question=question)
print(f"A: {result['answer']}")
print(f"\nSources: {len(result['sources'])}")
for i, src in enumerate(result['sources'][:3]):
    print(f"  [{i+1}] score={src.get('score', 0):.4f}")

## 5. Streaming 输出演示

In [ ]:
question = "融资和融券有什么区别？"
print(f"Q: {question}\n")
print("A: ", end="")

for chunk in pipeline.stream_query(question=question):
    if chunk["type"] == "answer":
        print(chunk["content"], end="", flush=True)
    elif chunk["type"] == "sources":
        sources = chunk["sources"]

print(f"\n\nSources: {len(sources)}")

## 6. Self-Correction 自我修正

In [ ]:
from src.correction.pipeline import SelfCorrectingPipeline

sc_pipeline = SelfCorrectingPipeline(
    pipeline=pipeline,
    config=config.self_correction,
    api_key=config.api_key,
    base_url=config.self_correction.verifier_base_url,
    model=config.self_correction.verifier_model,
)

question = "2024年中国GDP增速目标是多少？"
print(f"Q: {question}\n")

result = sc_pipeline.query(question=question)
print(f"A: {result['answer']}")

correction = result.get('correction')
if correction:
    print(f"\n--- Self-Correction Report ---")
    print(f"Passed: {correction.passed}")
    print(f"Confidence: {correction.confidence:.0%}")
    if correction.flagged_claims:
        print(f"Flagged: {correction.flagged_claims}")

## 7. Agent 多步推理

In [ ]:
question = "贵州茅台2024年的净利润率是多少？请检索并计算。"
print(f"Task: {question}\n")

result = pipeline.agent_query(task=question)
print(f"Answer: {result['answer']}")
print(f"\n--- Agent Steps ({len(result['steps'])}) ---")
for i, step in enumerate(result['steps']):
    print(f"\nStep {i+1}:")
    if step.get('thought'):
        print(f"  Thought: {step['thought']}")
    if step.get('action'):
        print(f"  Action: {step['action']}")
    if step.get('observation'):
        obs = step['observation'][:200]
        print(f"  Observation: {obs}...")

## 8. 知识图谱查询（GraphRAG）

In [ ]:
if config.graph.enabled:
    from src.graph.graph_store import create_graph_store

    graph = create_graph_store(config.graph)
    g_stats = graph.stats()
    print(f"Entities: {g_stats['num_entities']}")
    print(f"Triples: {g_stats['num_triples']}")

    # Query neighbors
    entity = "贵州茅台"
    triples = graph.query_neighbors(entity, max_depth=1)
    print(f"\nTriples for '{entity}':")
    for t in triples[:10]:
        print(f"  {t.head} --[{t.relation}]--> {t.tail}")
else:
    print("Graph not enabled. Set graph.enabled=true in config.yaml")